# Лабораторная работа №7. Маршрутизация: куда пойдёт пакет?

В Lab 006 мы выяснили, как префикс определяет connected route. Теперь между двумя Linux-хостами стоят два FRR-маршрутизатора. Исследуем connected (C), local (L), static (S), longest-prefix match, next hop, TTL и обратный путь.

Эксперименты: нет маршрута → добавляем static routes → пересылаем ICMP через два router hop → намеренно направляем один IP в Null0 через более специфичный /32 → диагностируем и восстанавливаем.

Меняем только учебные интерфейсы eth1/eth2. Служебный eth0 не трогаем.

## Схема

![Топология четырёх узлов](assets/routing.svg)

| Линк | Устройства | Сеть |
|---|---|---|
| LAN A | pc1:eth1 — r1:eth1 | 10.7.10.0/24 |
| Transit | r1:eth2 — r2:eth1 | 10.7.12.0/30 |
| LAN B | r2:eth2 — pc2:eth1 | 10.7.20.0/24 |

PC1/PC2 — Linux shell. R1/R2 — **FRR vtysh**. Команды show ip route и configure terminal выполняются только в терминале FRR, а Linux ip и ping — в терминале хоста.

## 1. Маршрут выбирается на каждом hop

Connected (C) появляется от адреса интерфейса, Local (L) — маршрут /32 к самому интерфейсу, Static (S) задаётся оператором. Default /0 — лишь последнее подходящее совпадение. Сначала ищем самый длинный prefix, совпадающий с destination, затем выбираем next hop и outgoing interface.

![Longest-prefix match: /32, /24, /0](assets/longest-prefix.svg)

Представьте на R1 две строки: сеть 10.7.20.0/24 через 10.7.12.2 и точный адрес 10.7.20.10/32 через Null0. Обе совпадают с destination 10.7.20.10, но победит самая специфичная. Даже если distance /24 ниже, это не делает его длиннее /32.

### Q1. Почему маршрут /24 к 10.7.20.10 выигрывает у /0? Что изменится после добавления маршрута /32 к тому же адресу?

`введите ваш ответ`

AI-REVIEW-PROMPT / REVIEW-ONLY. Проверь мой ответ на Q1 выше. Проверь логику рассуждения о маршрутизации, не раскрывай готовый ответ и не переписывай его. Отметь возможное упущение и задай 1–2 наводящих вопроса.

## 2. Настраиваем маршруты на PC1 и PC2

На pc1 (Linux):

~~~bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 10.7.10.10/24 dev eth1
ip -4 route replace 10.7.20.0/24 via 10.7.10.1 dev eth1
ip -4 route get 10.7.20.10 from 10.7.10.10
~~~

На pc2 (Linux):

~~~bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 10.7.20.10/24 dev eth1
ip -4 route replace 10.7.10.0/24 via 10.7.20.1 dev eth1
ip -4 route get 10.7.10.10 from 10.7.20.10
~~~

Оба компьютера знают только свой первый router hop. Это ещё не доказывает, что маршрутизаторы знают удалённые сети. Management default через eth0 не является учебным маршрутом.

## 3. Connected-сети на FRR-маршрутизаторах

В терминале **r1 — FRR vtysh**:

~~~text
configure terminal
interface eth1
 ip address 10.7.10.1/24
 exit
interface eth2
 ip address 10.7.12.1/30
 exit
end
show ip route
show ip route connected
~~~

В терминале **r2 — FRR vtysh**:

~~~text
configure terminal
interface eth1
 ip address 10.7.12.2/30
 exit
interface eth2
 ip address 10.7.20.1/24
 exit
end
show ip route
show ip route connected
~~~

R1 должен получить connected LAN A и transit, R2 — connected transit и LAN B. Записи L для адресов собственных интерфейсов — это отдельные маршруты /32. На данном этапе мы ещё НЕ добавляли static routes к удалённым LAN.

### Q2. Какие сети появляются автоматически на R1 и R2? Что означают C и L в show ip route, и почему это не S?

`введите ваш ответ`

AI-REVIEW-PROMPT / REVIEW-ONLY. Проверь мой ответ на Q2 выше. Проверь логику рассуждения о маршрутизации, не раскрывай готовый ответ и не переписывай его. Отметь возможное упущение и задай 1–2 наводящих вопроса.

## 4. Первый отказ: link исправен, но remote prefix неизвестен

На pc1:

~~~bash
ping -c 3 -W 1 10.7.20.10
traceroute -n -I -q 1 -w 1 -m 5 10.7.20.10
~~~

На R1 и R2:

~~~text
show ip route
~~~

R1 пока не имеет учебного маршрута к 10.7.20.0/24, а R2 — к 10.7.10.0/24. Не делайте вывод только по сообщению ping: это может быть ICMP unreachable или timeout. Сверьте routing table.

Запустите следующий capture на pc1:eth1 и во время записи повторите ping -c 3 -W 1 10.7.20.10.

### Q3. Почему правильный маршрут на pc1 через 10.7.10.1 сам по себе не обеспечивает доставку в LAN B? Что должны знать R1 и R2?

`введите ваш ответ`

AI-REVIEW-PROMPT / REVIEW-ONLY. Проверь мой ответ на Q3 выше. Проверь логику рассуждения о маршрутизации, не раскрывай готовый ответ и не переписывай его. Отметь возможное упущение и задай 1–2 наводящих вопроса.

In [ ]:
%%capture_traffic pc1:eth1 --filter "icmp or arp" --timeout 12 --packets 90 --save captures/routing-before-static.pcap
import time
print("На pc1: ping -c 3 -W 1 10.7.20.10")
time.sleep(10)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
before = PcapViewer("captures/routing-before-static.pcap", "icmp or arp", limit=60)
try:
    frames = before.packets()
    display(frames)
    if not frames.empty:
        print(before.packet(int(frames.iloc[0]["№"]), "protocols"))
    else:
        print("Пустой PCAP: повторите ping во время захвата.")
except Exception as exc:
    print("PCAP недоступен; проверьте время запуска и повторите:", exc)


## 5. Добавляем static routes в обоих направлениях

На R1 (vtysh):

~~~text
configure terminal
ip route 10.7.20.0/24 10.7.12.2
end
show ip route 10.7.20.0/24
~~~

На R2 (vtysh):

~~~text
configure terminal
ip route 10.7.10.0/24 10.7.12.1
end
show ip route 10.7.10.0/24
~~~

Внимательно сравните *destination prefix* и *next-hop IP*: next hop находится в transit /30 и принадлежит следующему маршрутизатору, а НЕ конечному компьютеру.

На pc1:

~~~bash
ping -c 3 -W 1 10.7.20.10
traceroute -n -I -q 1 -w 1 -m 5 10.7.20.10
~~~

На pc2:

~~~bash
ping -c 3 -W 1 10.7.10.10
~~~

Для traceroute используем -I, чтобы наблюдать ICMP Echo. Если отдельные hops показывают *, проверяйте реальные маршруты и ping — ICMP TTL-exceeded может фильтроваться.

### Q4. Чем destination 10.7.20.0/24 отличается от next hop 10.7.12.2? Почему нужен обратный маршрут R2→LAN A?

`введите ваш ответ`

AI-REVIEW-PROMPT / REVIEW-ONLY. Проверь мой ответ на Q4 выше. Проверь логику рассуждения о маршрутизации, не раскрывай готовый ответ и не переписывай его. Отметь возможное упущение и задай 1–2 наводящих вопроса.

## Что меняется на каждом router hop?

![Маршрутизаторы уменьшают TTL и заменяют Ethernet header](assets/packet-hop.svg)

IPv4 source и destination при обычной маршрутизации сохраняются, TTL уменьшается на каждом router hop; для следующего линка собирается новый Ethernet frame с адресами следующего L2-сегмента. ARP для next hop может не появиться, если neighbor cache уже заполнен.

Запустите захват на transit **r1:eth2**. Во время записи отправьте с pc1 ping -c 3 -W 1 10.7.20.10. Сравните IPv4 endpoints с PC1-стороной.

In [ ]:
%%capture_traffic r1:eth2 --filter "icmp or arp" --timeout 12 --packets 100 --save captures/routing-transit-success.pcap
import time
print("На pc1: ping -c 3 -W 1 10.7.20.10")
time.sleep(10)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
transit = PcapViewer("captures/routing-transit-success.pcap", "icmp or arp", limit=60)
try:
    packets = transit.packets()
    display(packets)
    icmp = packets[packets["Protocol"].astype(str).str.contains("ICMP", case=False, na=False)]
    if not icmp.empty:
        print(transit.packet(int(icmp.iloc[0]["№"]), "protocols"))
    else:
        print("Нет ICMP в захвате: проверьте static routes и время ping.")
except Exception as exc:
    print("PCAP недоступен:", exc)


## 6. Контролируемая поломка: host-route /32 в Null0

Оба /24 static routes исправны. На R1 добавьте **более специфичный маршрут только к PC2**:

~~~text
configure terminal
ip route 10.7.20.10/32 Null0
end
show ip route 10.7.20.0/24
show ip route 10.7.20.10/32
show ip route 10.7.20.10
~~~

**Null0** отбрасывает трафик без отключения портов. Маршрут /32 побеждает /24 для адреса 10.7.20.10.

На pc1 выполните ping -c 3 -W 1 10.7.20.10. Потеря возможна даже при наличии исправной /24 route в таблице R1. Результат нужно подтверждать выводом маршрутов: один лишь неответивший ping ничего не доказывает.

### Q5. Почему более конкретный /32 Null0 перехватывает трафик к PC2, хотя правильный /24 route ещё виден? Что покажет route lookup на R1?

`введите ваш ответ`

AI-REVIEW-PROMPT / REVIEW-ONLY. Проверь мой ответ на Q5 выше. Проверь логику рассуждения о маршрутизации, не раскрывай готовый ответ и не переписывай его. Отметь возможное упущение и задай 1–2 наводящих вопроса.

### Захват 3: отправленные Echo Request без успешного ответа

Запустите capture на pc1:eth1 и во время записи снова отправьте ping -c 3 -W 1 10.7.20.10. Посмотрите на Echo Request, Echo Reply и возможные ICMP errors. Пустой PCAP нельзя считать доказательством отсутствия трафика — всегда сверяйте таблицу маршрутов.

После опыта верните R1 в исходное рабочее состояние.

In [ ]:
%%capture_traffic pc1:eth1 --filter "icmp or arp" --timeout 12 --packets 100 --save captures/routing-blackhole.pcap
import time
print("На pc1: ping -c 3 -W 1 10.7.20.10")
time.sleep(10)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
blocked = PcapViewer("captures/routing-blackhole.pcap", "icmp or arp", limit=60)
try:
    packets = blocked.packets()
    display(packets)
    icmp = packets[packets["Protocol"].astype(str).str.contains("ICMP", case=False, na=False)]
    print("ICMP кадров:", len(icmp))
    if not icmp.empty:
        print(blocked.packet(int(icmp.iloc[0]["№"]), "protocols"))
except Exception as exc:
    print("PCAP недоступен:", exc)


## 7. Исправляем R1 и проверяем связность

На R1 (vtysh) удалите **только ошибочный /32**:

~~~text
configure terminal
no ip route 10.7.20.10/32 Null0
end
show ip route 10.7.20.10
show ip route 10.7.20.0/24
show running-config
~~~

Правильный /24 via 10.7.12.2 должен остаться. На R2 должен быть обратный static /24 via 10.7.12.1.

На pc1:

~~~bash
ping -c 3 -W 1 10.7.20.10
traceroute -n -I -q 1 -w 1 -m 5 10.7.20.10
~~~

На pc2:

~~~bash
ping -c 3 -W 1 10.7.10.10
~~~

Не перезапускайте маршрутизаторы, не меняйте eth0: задача — устранить ошибочный выбор более специфичного маршрута.

### Q6. После удаления /32 Null0 почему маршрут /24 снова обеспечивает доставку без изменения LAN и transit интерфейсов?

`введите ваш ответ`

AI-REVIEW-PROMPT / REVIEW-ONLY. Проверь мой ответ на Q6 выше. Проверь логику рассуждения о маршрутизации, не раскрывай готовый ответ и не переписывай его. Отметь возможное упущение и задай 1–2 наводящих вопроса.

## CHECK

Checker проверяет адреса PC1/PC2, host routes через правильный first hop, адреса интерфейсов FRR, наличие static /24 на обоих routers, отсутствие /32 Null0 и успешный ICMP **в обе стороны**.

Он оценивает итоговую работоспособность, а не количество ARP в capture или текст вашего ответа. Если CHECK падает: сначала show running-config и show ip route на R1/R2, затем ip route на PC1/PC2, затем ping.

## Итог

Каждый router самостоятельно выбирает longest matching destination prefix, затем next hop и egress interface. Connected /24 и transit /30 задают непосредственно подключённые сети, static routes связывают удалённые LAN и возвращают ответную дорогу. IP source/destination сохраняются, TTL уменьшается при пересылке, Ethernet header формируется отдельно для каждого линка.

Вы исправили два разных класса ошибок: отсутствующий static route и более конкретный /32 в Null0, не затрагивая физическую топологию или служебный eth0.